# Nommer les thèmes de la presse de 1914

BERTopic a décrit les thèmes de *L'Humanité* en 1914 par leurs mots-clés, sans leur donner de nom. Ici, un LLM local les nomme : le service par Ollama et la sortie structurée du cours, sur un vrai corpus.

Ce notebook va plus vite avec un GPU : *Exécution > Modifier le type d'exécution > GPU T4*. Sans GPU, il tourne quand même : un LLM plus petit (Qwen3-1.7B au lieu de Qwen3-4B), plus lent sur un processeur, ne nomme alors que les 5 plus grands thèmes.

## Données

Ollama s'installe et télécharge le LLM en arrière-plan.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
%pip install -q "bertopic<1"

import os
import subprocess

import torch

GPU = torch.cuda.is_available()
# Sur GPU, Qwen3-4B. Sur CPU, Qwen3-1.7B, deux fois plus rapide : un modèle « hybride »,
# qui réfléchit avant de répondre, sauf avec reasoning_effort="none" (dans les appels).
MODELE = "qwen3:4b-instruct-2507-q4_K_M" if GPU else "qwen3:1.7b"
# Sur GPU, 8 demandes à la fois ; sur CPU, une : le début commun des prompts (les
# consignes) n'est alors calculé qu'une fois.
parallele = "8" if GPU else "1"
env = {**os.environ, "OLLAMA_NUM_PARALLEL": parallele, "OLLAMA_KEEP_ALIVE": "-1"}
subprocess.Popen(
  ["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
telechargement = subprocess.Popen(
  f"sleep 5 && ollama pull {MODELE}",
  shell=True,
  stdout=subprocess.DEVNULL,
  stderr=subprocess.DEVNULL,
)

## Les thèmes

Cette cellule est donnée : elle refait les thèmes du TP « Les thèmes de la presse de 1914 », avec ses réglages. Ce sont les mêmes 20 000 passages de *L'Humanité*, leurs *embeddings* déjà calculés (Qwen3-Embedding-0.6B, tronqués à 512 dimensions), réduits à 5 dimensions par UMAP, avec des thèmes d'au moins 50 passages. Les passages hors thème sont ensuite réaffectés à leur thème le plus proche. BERTopic prévient alors que cette réaffectation doit être la dernière étape : c'est le cas ici. Le calcul prend quelques minutes, pendant que le LLM se télécharge.

In [ ]:
import numpy as np
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.fr.stop_words import STOP_WORDS
from umap import UMAP

passages = pd.read_parquet("passages.parquet")
embeddings = np.load("embeddings-512.npy").astype(np.float32)

# L'Humanité seulement (Le Figaro n'a que 24 numéros), et un échantillon de 20 000
# passages pour que le calcul tienne en quelques minutes.
humanite = passages[passages.newspaper == "L'Humanité"].sample(20_000, random_state=0)
textes = humanite.text.tolist()
vecteurs = embeddings[humanite.index]

STOP_WORDS = sorted(STOP_WORDS | {"qu", "mm", "mme"})
modele_themes = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
  ),
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
  min_topic_size=50,
)
themes, _ = modele_themes.fit_transform(textes, vecteurs)

# Les passages hors thème rejoignent leur thème le plus proche.
nouveaux_themes = modele_themes.reduce_outliers(
  textes, themes, strategy="embeddings", embeddings=vecteurs
)
modele_themes.update_topics(
  textes,
  topics=nouveaux_themes,
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
)
infos = modele_themes.get_topic_info().sort_values("Count", ascending=False)
infos[["Topic", "Count", "Representation"]].head(10)

## Nommer les thèmes avec un LLM

Des mots-clés décrivent un thème, mais ne le nomment pas. Pour chacun des `N_THEMES` plus grands thèmes (20 ; sans GPU, 5), donnez au LLM ses 10 mots-clés et ses 3 passages représentatifs (leurs 300 premiers caractères), et demandez-lui un nom court, de 3 à 6 mots, en français. Le client de la frame « Servir un modèle soi-même » est prêt : appelez `client.chat.completions.parse` pour une sortie contrainte au format `NomTheme` (frame « Contraindre la sortie » ; rappelez ce format dans le prompt : le modèle ne voit pas le schéma). Lancez les demandes en parallèle, installez les noms avec `modele_themes.set_topic_labels(noms)` (un dictionnaire thème → nom), et affichez `get_topic_info()`. Les noms sont-ils justes ? Lequel corrigeriez-vous à la main ?

**Documentation utile**

- [`chat.completions.parse`](https://github.com/openai/openai-python/blob/main/helpers.md#auto-parsing-response-content-with-pydantic-models)
- [Les modèles Pydantic](https://docs.pydantic.dev/latest/concepts/models/)
- [L'API OpenAI d'Ollama](https://docs.ollama.com/api/openai-compatibility)
- [`ThreadPoolExecutor.map`](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map)
- [`BERTopic.get_topic`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic)
- [`BERTopic.get_representative_docs`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_representative_docs)
- [`BERTopic.get_topic_info`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic_info)
- [`BERTopic.set_topic_labels`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.set_topic_labels)

In [ ]:
telechargement.wait()  # le modèle est téléchargé

from concurrent.futures import ThreadPoolExecutor

from openai import OpenAI
from pydantic import BaseModel

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# Un premier appel charge le modèle (sur le GPU s'il y en a un), où il reste : avant les calculs en parallèle.
client.chat.completions.create(
  model=MODELE, messages=[{"role": "user", "content": "Bonjour"}], max_tokens=1
)
print(MODELE, "prêt")


class NomTheme(BaseModel):
  nom: str


# Les thèmes à nommer : 20 sur GPU ; sans GPU, 5 (une à deux minutes).
N_THEMES = 20 if GPU else 5

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `modele_themes.get_topic(theme)[:10]` : les 10 premiers couples (mot-clé, poids) ; `modele_themes.get_representative_docs(theme)[:3]` : 3 passages, dont on garde `texte[:300]`
- le prompt rappelle le format : `Réponds par un objet JSON : {"nom": "..."}` ; le schéma contraint la sortie, mais le modèle ne le voit pas, et il lui arrive de s'égarer en espaces avant le JSON
- `client.chat.completions.parse(model=MODELE, messages=[{"role": "user", "content": prompt}], response_format=NomTheme, temperature=0, reasoning_effort="none", max_tokens=100)` ; le nom est dans `reponse.choices[0].message.parsed.nom`
- `reasoning_effort="none"` : le modèle répond sans phase de réflexion (sans GPU, Qwen3-1.7B en aurait une, qui multiplie le temps de réponse)
- les `N_THEMES` plus grands thèmes : après la réaffectation, il n'y a plus de thème −1, et `get_topic_info()` suit l'ordre des numéros : triez-le avec `sort_values("Count", ascending=False)`, puis `infos.Topic[:N_THEMES].tolist()`
- `ThreadPoolExecutor(8)` : 8 appels en parallèle ; `pool.map(nommer_theme, plus_grands)` renvoie les noms dans l'ordre ; `dict(zip(plus_grands, noms))`
- `modele_themes.set_topic_labels(noms)` : les noms apparaissent dans la colonne `CustomName` de `get_topic_info()`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def nommer_theme(theme: int) -> str:
  mots_cles = ", ".join(mot for mot, _ in modele_themes.get_topic(theme)[:10])
  exemples = "\n\n".join(
    texte[:300] for texte in modele_themes.get_representative_docs(theme)[:...]
  )
  reponse = client.chat.completions.parse(
    model=MODELE,
    messages=[
      {
        "role": "user",
        "content": (
          "Voici un thème de la presse française de 1914, trouvé automatiquement.\n"
          f"Ses mots-clés : {mots_cles}\n\nTrois passages représentatifs :\n{exemples}\n\n"
          "Donne-lui un nom court et précis, de 3 à 6 mots, en français.\n"
          'Réponds par un objet JSON : {"nom": "..."}'
        ),
      }
    ],
    response_format=...,
    temperature=0,
    reasoning_effort="none",
    max_tokens=100,
  )
  return reponse.choices[0].message.parsed.nom


infos = modele_themes.get_topic_info().sort_values("Count", ascending=False)
plus_grands = infos.Topic[:...].tolist()
with ThreadPoolExecutor(8) as pool:
  noms = dict(zip(plus_grands, pool.map(..., plus_grands)))
modele_themes.set_topic_labels(...)
infos = modele_themes.get_topic_info().sort_values("Count", ascending=False)
infos[["Topic", "Count", "CustomName", "Representation"]].head(N_THEMES)
```

### Solution

In [ ]:
def nommer_theme(theme: int) -> str:
  mots_cles = ", ".join(mot for mot, _ in modele_themes.get_topic(theme)[:10])
  exemples = "\n\n".join(
    texte[:300] for texte in modele_themes.get_representative_docs(theme)[:3]
  )
  reponse = client.chat.completions.parse(
    model=MODELE,
    messages=[
      {
        "role": "user",
        "content": (
          "Voici un thème de la presse française de 1914, trouvé automatiquement.\n"
          f"Ses mots-clés : {mots_cles}\n\nTrois passages représentatifs :\n{exemples}\n\n"
          "Donne-lui un nom court et précis, de 3 à 6 mots, en français.\n"
          # Le schéma contraint la sortie, mais le modèle ne le voit pas : sans ce rappel,
          # il lui arrive de s'égarer en espaces avant le JSON.
          'Réponds par un objet JSON : {"nom": "..."}'
        ),
      }
    ],
    response_format=NomTheme,
    temperature=0,
    reasoning_effort="none",
    max_tokens=100,
  )
  return reponse.choices[0].message.parsed.nom


infos = modele_themes.get_topic_info().sort_values("Count", ascending=False)
plus_grands = infos.Topic[:N_THEMES].tolist()
with ThreadPoolExecutor(8) as pool:
  noms = dict(zip(plus_grands, pool.map(nommer_theme, plus_grands)))
modele_themes.set_topic_labels(noms)
infos = modele_themes.get_topic_info().sort_values("Count", ascending=False)
infos[["Topic", "Count", "CustomName", "Representation"]].head(N_THEMES)

# BERTopic sait aussi le faire seul : representation_model=bertopic.representation.OpenAI(client,
# model=MODELE), avec n'importe quel client compatible OpenAI. Les noms se relisent toujours :
# un LLM nomme bien un thème net, mal un thème fourre-tout.